In [27]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ['GROQ_API_KEY']=os.getenv("GROQ_API_KEY")

os.environ['LANGCHAIN_API_KEY']=os.getenv("LANGCHAIN_API_KEY")

In [28]:
from langchain_community.document_loaders import WebBaseLoader

loader=WebBaseLoader(
    "https://www.yakrontech.in/"
)
data=loader.load()

In [29]:
data

[Document(metadata={'source': 'https://www.yakrontech.in/', 'title': 'Yakron Technologies | Web, App & AI Development Company in Ratnagiri', 'description': 'Yakron Technologies builds websites, dynamic web applications and AI-powered systems for growing businesses — from a simple business page to a full ERP and automation platform. Based in Ratnagiri, Maharashtra.', 'language': 'en'}, page_content="\n\n\n\n\nYakron Technologies | Web, App & AI Development Company in Ratnagiri\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nWhy Yakron\nServices\nOur Work\nInternships\nProcess\nLet's Talk\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nNow serving businesses in Ratnagiri & beyond, at honest pricing\nBuild.Automate.Scale.\nYakron Technologies designs and builds websites, dynamic web applications and AI-powered systems — from a simple business page to a full automation platform.\n\nChat on WhatsApp\nSee our work →\n\n\n5+Systems shipped\n100%Client satisfaction\nEnd-to-endDesign → Depl

In [30]:
# split the data into parts

from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50)

splitted_text=splitter.split_documents(data)

In [31]:
splitted_text

[Document(metadata={'source': 'https://www.yakrontech.in/', 'title': 'Yakron Technologies | Web, App & AI Development Company in Ratnagiri', 'description': 'Yakron Technologies builds websites, dynamic web applications and AI-powered systems for growing businesses — from a simple business page to a full ERP and automation platform. Based in Ratnagiri, Maharashtra.', 'language': 'en'}, page_content="Yakron Technologies | Web, App & AI Development Company in Ratnagiri\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nWhy Yakron\nServices\nOur Work\nInternships\nProcess\nLet's Talk\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nNow serving businesses in Ratnagiri & beyond, at honest pricing\nBuild.Automate.Scale.\nYakron Technologies designs and builds websites, dynamic web applications and AI-powered systems — from a simple business page to a full automation platform.\n\nChat on WhatsApp\nSee our work →"),
 Document(metadata={'source': 'https://www.yakrontech.in/', 'title': 'Yakron Te

In [32]:
# embeddings(vector)

from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings=HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1577.83it/s]


In [33]:
# vectore store

from langchain_community.vectorstores import Chroma

vector_store=Chroma.from_documents(
    splitted_text,
    embedding=embeddings
)

In [34]:
# create doc chain and retrival chain

from langchain_core.prompts import ChatPromptTemplate

prompt=ChatPromptTemplate.from_messages([
    ("system","you are a helpful assistant who provide answers based on the given context only context:{context}"),
    ("user","{input}")
])



In [35]:
# llm

from langchain_groq import ChatGroq

llm=ChatGroq(
    model='openai/gpt-oss-120b',
    temperature=0
)

In [36]:
# vector store as ertriver

retriver=vector_store.as_retriever(search_kwargs={"k": 5})

In [37]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain

document_chain=create_stuff_documents_chain(llm,prompt)
retrival_chain=create_retrieval_chain(retriver,document_chain)

In [38]:
response=retrival_chain.invoke({"input":"what yakron technologies serve?"})

print(response['answer'])

Yakron Technologies serves businesses—primarily in Ratnagiri and beyond—by designing and building:

* **Websites** (from simple business pages)  
* **Dynamic web applications**  
* **AI‑powered systems** and automation platforms (e.g., AI‑driven ERP, campus‑management solutions, face‑recognition attendance, online proctored exams, AI chat assistants, etc.)

In short, they provide end‑to‑end web, app and AI development services for companies that need digital, automated, and intelligent solutions.


In [39]:
response2=retrival_chain.invoke({"input":"i want to contact them how do i do?"})

print(response2['answer'])

You can reach Yakron Technologies directly through the contact details shown on their site:

**Phone / WhatsApp**  
- **Number:** +91 90282 86738  
- Simply dial the number to call, or open WhatsApp, add the same number, and send them a message.

**How to use each option**

| Method | Steps |
|--------|-------|
| **Call** | 1. Open your phone’s dialer.<br>2. Enter **+91 90282 86738**.<br>3. Press the call button and speak with a representative. |
| **WhatsApp** | 1. Open WhatsApp on your phone (or WhatsApp Web on a computer).<br>2. Tap “New Chat” → “New Contact” and save **+91 90282 86738** if you haven’t already.<br>3. Open the chat and type your inquiry (e.g., “Hi, I’d like to know more about your services”). |
| **Website “Get in touch” section** | The site also has a “Get in touch” link in the navigation bar. Clicking it will usually bring up a contact form or repeat the same phone/WhatsApp details. |

That’s all you need—just pick the method that’s most convenient for you and reac

In [40]:
# now addding a chat history to this 


from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder
)

from langchain_classic.chains import (
    create_history_aware_retriever,
    create_retrieval_chain
)

from langchain_classic.chains.combine_documents import (
    create_stuff_documents_chain
)

contextualize_q_system_prompt = (
    "Given a chat history and the latest user question "
    "which might reference context in the chat history, "
    "formulate a standalone question which can be understood "
    "without the chat history. Do NOT answer the question, "
    "just reformulate it if needed and otherwise return it as is."
)

# Prompt to understand previous conversation
contextualize_q_prompt = ChatPromptTemplate.from_messages([
    
    ("system",contextualize_q_system_prompt),
    
    MessagesPlaceholder("chat_history"),
    
    ("human", "{input}")
])

history_aware_retriever=create_history_aware_retriever(llm,retriver,contextualize_q_prompt)



In [41]:
# Normal answer prompt
qa_prompt = ChatPromptTemplate.from_messages([
    
    ("system",
     """Answer the question using the provided context only.

     Context:
     {context}"""),
    
    ("human", "{input}")
])


# Document chain
document_chain = create_stuff_documents_chain(
    llm,
    qa_prompt
)

In [42]:
retrieval_chain = create_retrieval_chain(
    history_aware_retriever,
    document_chain
)

In [43]:
chat_history=[]

In [44]:
chat_history

[]

In [53]:

input_question=input("enter your question: ")

response=retrieval_chain.invoke({
    "input":input_question,
    "chat_history": chat_history
})

print(response["answer"])



You can reach Yakron Technologies directly through the contact details they provide:

- **WhatsApp:** +91 90282 86738  (you can “Chat on WhatsApp” from their site)  
- **Phone call:** +91 90282 86738  

Both numbers are the same, so you can either start a WhatsApp conversation or give them a call. Their office is located in **Ratnagiri, Maharashtra**.  

Just open WhatsApp, add the number, and send a message, or dial the number from your phone to speak with them.


In [54]:
chat_history.extend([
    ("human",input_question),
    ("ai",response["answer"])
])

In [55]:
chat_history

[('human', 'whi is yakron?'),
 ('ai',
  'Yakron\u202fTechnologies is a **web, app and AI development company** based in Ratnagiri, Maharashtra. It designs and builds websites, dynamic web applications and AI‑powered systems—ranging from simple business pages to full automation platforms—for businesses in Ratnagiri and beyond.'),
 ('human', 'what they serve?'),
 ('ai',
  'Yakron Technologies\u202fserves **businesses in Ratnagiri and beyond**. They provide web, app and AI development services—building everything from simple business websites to full‑scale automation platforms and AI‑powered systems for their clients.'),
 ('human', 'if i want to connect with them then how do i do??'),
 ('ai',
  'You can reach Yakron Technologies directly through the contact details they provide:\n\n- **WhatsApp:**\u202f+91\u202f90282\u202f86738\u202f\u202f(you can “Chat on WhatsApp” from their site)  \n- **Phone call:**\u202f+91\u202f90282\u202f86738  \n\nBoth numbers are the same, so you can either start